# Best Buy Canada Review Sentiment Analysis

This notebook documents the collection, validation, sentiment-scoring, export, and stakeholder-insight workflow. Collection is **off by default**. A local 50-review sample is available at `outputs/asus_cm14_bestbuy_reviews_sentiment.csv`; it was captured from visible public review cards and is not synthetic. Before collecting additional data, check current terms and `https://www.bestbuy.ca/robots.txt`; use direct product pages and visible controls only. Never bypass a block, CAPTCHA, or rate limit.

The sample page displayed 434 reviews, and 50 distinct cards were loaded through the visible Show More control. Important: some review text mentions ASUS CX14 / Intel Celeron / 64GB hardware, whereas the listing is CM14 / MediaTek / 128GB. Treat the analysis as a review-page sample, not verified CM14-specific evidence; a matching product page could not be verified because Best Buy search returned 403. One visible card had no source title, so the display title is derived from review text and marked in `title_source`.

## 1. Setup and shared application functions

Install the dependencies from `requirements.txt` in the active Python environment before running the cells.

In [ ]:
from pathlib import Path
import pandas as pd

from src.bestbuy_reviews import ScrapeSettings, scrape_reviews
from src.data_io import load_review_csv
from src.sentiment import analyze_sentiment, build_insights, recommendations

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Ready. Scraping is not started by importing the modules.')

## 2. Optional public review collection

Set `RUN_SCRAPE = True` only after confirming the current site terms, robots.txt permission, and your authorization. The collector restricts URLs to `/en-ca/product/`, checks robots.txt before browsing, uses visible controls, applies a polite pause and maximum page count, and will stop if the visible layout is not recognized. It does not access private APIs or circumvent access controls.

In [ ]:
PRODUCT_URL = 'https://www.bestbuy.ca/en-ca/product/asus-cm14-14-chromebook-mediatek-kompanio-540-4gb-lpddr5x-128gb-emmc-chromeos-pure-grey/19826852'
SORT_VIEWS = ['Most Helpful', 'Newest', 'Highest Rating', 'Lowest Rating', 'Most Relevant']
MAX_PAGES_PER_SORT = 6
RUN_SCRAPE = False  # Change only after reviewing current policies and permissions
POLICY_CONFIRMED = False  # Explicit operator confirmation is required

scraped_reviews = pd.DataFrame()
if RUN_SCRAPE:
    if not POLICY_CONFIRMED:
        raise PermissionError('Confirm current terms, robots.txt, and authorization before scraping.')
    scraped_reviews = scrape_reviews(
        ScrapeSettings(
            product_url=PRODUCT_URL,
            sort_options=tuple(SORT_VIEWS),
            max_pages_per_sort=MAX_PAGES_PER_SORT,
            pause_seconds=2.0,
        ),
        policy_confirmed=POLICY_CONFIRMED,
    )
    print(f'Collected {len(scraped_reviews)} unique reviews. Check whether this meets the 50-review minimum.')
else:
    print('Collection skipped. Provide an authorized CSV below to analyze real review data.')

## 3. Load and validate review data

Use the scrape result or a UTF-8 CSV you are authorized to analyze. The required text field is `review_text`; expected metadata fields include `primary_key`, `title`, `date`, `rating`, `source`, and `reviewer_name`. Do not treat synthetic examples as scraped output.

In [ ]:
CSV_PATH = Path('outputs/asus_cm14_bestbuy_reviews_sentiment.csv')  # Replace with another authorized CSV if needed
if not scraped_reviews.empty:
    reviews = scraped_reviews.copy()
elif CSV_PATH.exists():
    reviews = load_review_csv(CSV_PATH)
else:
    reviews = pd.DataFrame()

if not reviews.empty:
    if 'review_text' not in reviews.columns:
        raise ValueError('Input CSV must contain a review_text column.')
    reviews['review_text'] = reviews['review_text'].fillna('').astype(str)
    reviews = reviews.loc[reviews['review_text'].str.strip().ne('')].copy()
    print(f'{len(reviews)} usable reviews loaded; 50-review threshold met: {len(reviews) >= 50}')
    missing_fields = [field for field in ['primary_key', 'title', 'date', 'rating', 'source', 'reviewer_name'] if field not in reviews.columns]
    print('Missing metadata columns:', missing_fields or 'none')
    if 'title_source' in reviews.columns:
        print('Titles derived from review text:', int(reviews['title_source'].eq('generated_from_review_text').sum()))
else:
    print('No authorized review data found. Set CSV_PATH or run the opt-in collection cell.')

## 4. Sentiment analysis and review categories

VADER produces a text-only compound score from -1 to +1 and a raw `text_sentiment` label. For the final `sentiment` label, reviews rated 1–2 stars are treated as Negative to avoid a known lexicon failure where strongly dissatisfied wording can score positive. Other reviews follow the VADER thresholds (>= 0.05 Positive, <= -0.05 Negative, otherwise Neutral). The rating-aware category is shown separately. This is exploratory; validate a hand-coded sample before decisions.

In [ ]:
if not reviews.empty:
    analyzed = analyze_sentiment(reviews)
    display(analyzed.head(10))
    print(analyzed['sentiment'].value_counts(dropna=False))
else:
    analyzed = pd.DataFrame()
    print('Sentiment scoring skipped until review data is supplied.')

## 5. Export CSV and Excel results

Exports are generated only from supplied or collected data. No empty or synthetic dataset is presented as a Best Buy scrape.

In [ ]:
if not analyzed.empty:
    insights = build_insights(analyzed)
    csv_output = OUTPUT_DIR / 'asus_cm14_bestbuy_reviews_sentiment.csv'
    excel_output = OUTPUT_DIR / 'asus_cm14_bestbuy_reviews_sentiment.xlsx'
    analyzed.to_csv(csv_output, index=False, encoding='utf-8-sig')
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        analyzed.to_excel(writer, index=False, sheet_name='Reviews')
        pd.DataFrame([{'metric': k, 'value': str(v)} for k, v in insights.items()]).to_excel(
            writer, index=False, sheet_name='Insights'
        )
        pd.DataFrame([{'recommendation': item} for item in recommendations(insights)]).to_excel(
            writer, index=False, sheet_name='Recommendations'
        )
        pd.DataFrame([{'metric': 'scope', 'value': '50 visible-card sample; not all 434 listing reviews'}, {'metric': 'model_fit', 'value': 'Review text contains CX14/Intel Celeron/64GB references on the CM14/MediaTek/128GB listing; no matching alternative page was verifiable after Best Buy search returned 403'}, {'metric': 'title_provenance', 'value': 'Source title for one card was absent; display title is derived from review text and marked in title_source'}, {'metric': 'authorization', 'value': 'Confirm current terms and authorization before reuse or redistribution'}]).to_excel(
            writer, index=False, sheet_name='Collection'
        )
    print(f'Wrote {csv_output} and {excel_output}')
else:
    print('No export created.')

## 6. Stakeholder insight summary

Term counts are hypotheses, not causal explanations. Validate themes with review reading and operational data (returns, support contacts, product variants) before recommending a product change.

In [ ]:
if not analyzed.empty:
    insights = build_insights(analyzed)
    print(f"Reviews: {insights['review_count']}")
    print(f"Average rating: {insights['average_rating']}")
    print(f"Positive share: {insights['positive_share']:.1%}")
    print(f"Negative share: {insights['negative_share']:.1%}")
    print('Potential positive themes:', insights['top_positive_terms'])
    print('Potential negative themes:', insights['top_negative_terms'])
    for action in recommendations(insights):
        print('-', action)
else:
    print('No data available for stakeholder insights.')

## Limitations and responsible use

- Re-check Best Buy robots.txt and terms immediately before any run. If the page or policy is unclear, stop and request permission.
- The collector uses the site's visible Reject All option to decline optional cookies when available; it never accepts optional cookies automatically. If that option is missing, collection stops; this app does not provide a manual consent flow for its browser session. Do not use proxy rotation, CAPTCHA bypass, fake identities, login automation, or rate-limit evasion.
- Selenium requires a compatible Chrome browser. The page's visible labels and markup may change; errors indicate that collection is incomplete, not that the product has no reviews.
- Deduplication across sort views and maximum pages can leave a partial sample. Compare the output with the public count; the app warns when fewer than 50 reviews are available.
- Reviewer names may be personal data. Minimize, pseudonymize, and retain only as long as needed.
- See `reports/stakeholder_report.md` for scraping challenges and stakeholder recommendations.